# Arm A — check OmniGen2 prompting levers (read-only)
Prints the pipeline __call__ signature, the dual-CFG combination, and the negative-prompt handling, so the prompting variants map onto real levers. No model load, no GPU.

## 1. Clone + locate pipeline

In [1]:
# clone OmniGen2, locate the generation pipeline
import os, subprocess
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/VectorSpaceLab/OmniGen2", REPO],
                   check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
PIPE = f"{REPO}/omnigen2/pipelines/omnigen2/pipeline_omnigen2.py"
src = open(PIPE).read()
print("pipeline:", PIPE, "| exists:", os.path.exists(PIPE), "| lines:", src.count(chr(10)))

pipeline: /content/OmniGen2/omnigen2/pipelines/omnigen2/pipeline_omnigen2.py | exists: True | lines: 773


## 2. __call__ signature (the levers)

In [2]:
# the __call__ signature = every training-free lever (defaults shown)
i = src.find("def __call__")
print(src[i: src.find("):", i) + 2])

def __call__(
        self,
        prompt: Optional[Union[str, List[str]]] = None,
        negative_prompt: Optional[Union[str, List[str]]] = None,
        prompt_embeds: Optional[torch.FloatTensor] = None,
        negative_prompt_embeds: Optional[torch.FloatTensor] = None,
        prompt_attention_mask: Optional[torch.LongTensor] = None,
        negative_prompt_attention_mask: Optional[torch.LongTensor] = None,
        max_sequence_length: Optional[int] = None,
        callback_on_step_end_tensor_inputs: Optional[List[str]] = None,
        input_images: Optional[List[PIL.Image.Image]] = None,
        num_images_per_prompt: int = 1,
        height: Optional[int] = None,
        width: Optional[int] = None,
        max_pixels: int = 1024 * 1024,
        max_input_image_side_length: int = 1024,
        align_res: bool = True,
        num_inference_steps: int = 28,
        text_guidance_scale: float = 4.0,
        image_guidance_scale: float = 1.0,
        cfg_range: Tuple[float, float] 

## 3. Dual-CFG — image_guidance (SC) + text_guidance (PF) + negative prompt (uncond)

In [3]:
# dual-CFG: how negative prompt (uncond), image_guidance_scale (SC), text_guidance_scale (PF) combine
i = src.find("model_pred = model_pred_uncond")
print("----- denoising guidance combination -----")
print(src[i: i + 320])
print()
print("reading: uncond = negative-prompt branch;")
print("         + image_guidance_scale * (ref  - uncond)  -> pushes toward the reference subject (SC)")
print("         + text_guidance_scale  * (full - ref)     -> pushes toward the instruction (PF)")

----- denoising guidance combination -----
model_pred = model_pred_uncond + image_guidance_scale * (model_pred_ref - model_pred_uncond) + \
                        text_guidance_scale * (model_pred - model_pred_ref)
                elif text_guidance_scale > 1.0:
                    if enable_taylorseer:
                        self.transformer.cache_dic = mode

reading: uncond = negative-prompt branch;
         + image_guidance_scale * (ref  - uncond)  -> pushes toward the reference subject (SC)
         + text_guidance_scale  * (full - ref)     -> pushes toward the instruction (PF)


## 4. Negative-prompt handling (encode_prompt)

In [4]:
# negative_prompt handling: defaults to "", chat-templated + Qwen-encoded like the positive prompt
i = src.find("if do_classifier_free_guidance and negative_prompt_embeds is None")
print("----- encode_prompt: negative prompt -> uncond embeds -----")
print(src[i: i + 650])

----- encode_prompt: negative prompt -> uncond embeds -----
if do_classifier_free_guidance and negative_prompt_embeds is None:
            negative_prompt = negative_prompt if negative_prompt is not None else ""

            # Normalize str to list
            negative_prompt = batch_size * [negative_prompt] if isinstance(negative_prompt, str) else negative_prompt
            negative_prompt = [self._apply_chat_template(_negative_prompt) for _negative_prompt in negative_prompt]

            if prompt is not None and type(prompt) is not type(negative_prompt):
                raise TypeError(
                    f"`negative_prompt` should be the same type to `prompt`, but got {type(negative_prompt)} !="


## 5. Pipeline defaults vs baseline engine settings

In [5]:
# confirm the exact defaults the pipeline ships (AST-parsed from the __call__ signature)
import ast
tree = ast.parse(src)
defaults = {}
for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef) and node.name == "__call__":
        a = node.args
        off = len(a.args) - len(a.defaults)
        for i, dft in enumerate(a.defaults):
            defaults[a.args[off + i].arg] = ast.unparse(dft)
        for arg, dft in zip(a.kwonlyargs, a.kw_defaults):
            if dft is not None:
                defaults[arg.arg] = ast.unparse(dft)
        break

for p in ["text_guidance_scale", "image_guidance_scale", "num_inference_steps", "cfg_range", "align_res", "max_pixels"]:
    print("%-22s pipeline default = %s" % (p, defaults.get(p, "?")))
print()
print("baseline engine used (= OmniGen2 inference.py defaults): text_guidance_scale=5.0, image_guidance_scale=2.0, steps=50, cfg_range=(0,1), align_res=False, 512x512")

text_guidance_scale    pipeline default = 4.0
image_guidance_scale   pipeline default = 1.0
num_inference_steps    pipeline default = 28
cfg_range              pipeline default = (0.0, 1.0)
align_res              pipeline default = True
max_pixels             pipeline default = 1024 * 1024

baseline engine used (= OmniGen2 inference.py defaults): text_guidance_scale=5.0, image_guidance_scale=2.0, steps=50, cfg_range=(0,1), align_res=False, 512x512


## 6. inference.py recommended defaults (what actually runs)

In [6]:
import os, re
inf = f"{REPO}/inference.py"
if os.path.exists(inf):
    isrc = open(inf).read()
    print("\n--- inference.py recommended defaults (what actually runs) ---")
    for flag in ["text_guidance_scale", "image_guidance_scale", "num_inference_step"]:
        m = re.search(r'"--%s".*?default\s*=\s*([^,\s]+)' % re.escape(flag), isrc, re.S)
        print("%-22s = %s" % (flag, m.group(1) if m else "?"))


--- inference.py recommended defaults (what actually runs) ---
text_guidance_scale    = 5.0
image_guidance_scale   = 2.0
num_inference_step     = 50
